# Analise dos Logs de Acesso RFID - HydraSensor

Este notebook responde as perguntas solicitadas no PDF do projeto usando os logs exportados pelo backend.

Fonte dos dados: `rfid_access_log.csv` ou CSV obtido pelo endpoint `GET /v1/access-events/export.csv`.


## 1. Carregar Dados


In [ ]:
from pathlib import Path

import pandas as pd

CSV_PATH = Path('rfid_access_log.csv')

df = pd.read_csv(CSV_PATH)
df['read_at'] = pd.to_datetime(df['read_at'])
df['created_at'] = pd.to_datetime(df['created_at'])
df['date'] = df['read_at'].dt.date

df.head()


## 2. Escolher Dia de Analise


In [ ]:
# Altere esta data se quiser analisar outro dia.
DATA_ANALISE = df['date'].max()
dia = df[df['date'] == DATA_ANALISE].copy()

print(f'Data analisada: {DATA_ANALISE}')
print(f'Total de eventos no dia: {len(dia)}')
dia[['tag_id', 'collaborator_name', 'event_type', 'authorized', 'origin', 'read_at']]


## 3. Quantas Pessoas Entraram na Sala no Dia


In [ ]:
entradas = dia[dia['event_type'] == 'entrada']

print(f'Quantidade de entradas registradas: {len(entradas)}')
print(f'Quantidade de pessoas distintas que entraram: {entradas["collaborator_id"].nunique()}')

entradas[['collaborator_id', 'collaborator_name', 'registration', 'read_at']]


## 4. Quantas Pessoas Sairam da Sala no Dia


In [ ]:
saidas = dia[dia['event_type'] == 'saida']

print(f'Quantidade de saidas registradas: {len(saidas)}')
print(f'Quantidade de pessoas distintas que sairam: {saidas["collaborator_id"].nunique()}')

saidas[['collaborator_id', 'collaborator_name', 'registration', 'read_at']]


## 5. Tempo de Permanencia por Colaborador


In [ ]:
def calcular_permanencia(grupo):
    grupo = grupo.sort_values('read_at')
    entrada_aberta = None
    intervalos = []

    for _, evento in grupo.iterrows():
        if evento['event_type'] == 'entrada':
            entrada_aberta = evento
        elif evento['event_type'] == 'saida' and entrada_aberta is not None:
            duracao = evento['read_at'] - entrada_aberta['read_at']
            intervalos.append({
                'collaborator_id': entrada_aberta['collaborator_id'],
                'collaborator_name': entrada_aberta['collaborator_name'],
                'registration': entrada_aberta['registration'],
                'entrada': entrada_aberta['read_at'],
                'saida': evento['read_at'],
                'duracao': duracao,
                'duracao_segundos': duracao.total_seconds(),
            })
            entrada_aberta = None

    return intervalos

eventos_presenca = dia[
    (dia['authorized'] == 1) &
    (dia['event_type'].isin(['entrada', 'saida'])) &
    (dia['collaborator_id'].notna())
]

intervalos = []
for _, grupo in eventos_presenca.groupby('collaborator_id'):
    intervalos.extend(calcular_permanencia(grupo))

permanencia_intervalos = pd.DataFrame(intervalos)

if permanencia_intervalos.empty:
    print('Nenhum intervalo completo de entrada/saida encontrado no dia.')
else:
    permanencia_total = (
        permanencia_intervalos
        .groupby(['collaborator_id', 'collaborator_name', 'registration'], as_index=False)['duracao_segundos']
        .sum()
    )
    permanencia_total['duracao_total'] = pd.to_timedelta(permanencia_total['duracao_segundos'], unit='s')
    display(permanencia_intervalos)
    display(permanencia_total)


## 6. Tentativas de Acesso Negado no Dia


In [ ]:
acessos_negados = dia[dia['event_type'] == 'acesso_negado']

print(f'Tentativas de acesso negado: {len(acessos_negados)}')
acessos_negados[['tag_id', 'collaborator_name', 'registration', 'message', 'read_at']]


## 7. Tentativas de Invasao no Dia


In [ ]:
invasoes = dia[dia['event_type'] == 'invasao']

print(f'Tentativas de invasao: {len(invasoes)}')
invasoes[['tag_id', 'collaborator_name', 'message', 'read_at']]


## 8. Ranking de Colaboradores Nao Autorizados


In [ ]:
ranking_nao_autorizados = (
    acessos_negados
    .groupby(['collaborator_id', 'collaborator_name', 'registration'], dropna=False)
    .size()
    .reset_index(name='tentativas')
    .sort_values('tentativas', ascending=False)
)

ranking_nao_autorizados


## 9. Resumo Final


In [ ]:
resumo = {
    'data': str(DATA_ANALISE),
    'entradas': len(entradas),
    'pessoas_distintas_que_entraram': int(entradas['collaborator_id'].nunique()),
    'saidas': len(saidas),
    'pessoas_distintas_que_sairam': int(saidas['collaborator_id'].nunique()),
    'acessos_negados': len(acessos_negados),
    'invasoes': len(invasoes),
}

resumo
